In [4]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

N_QUBITS = 8
df_data = pd.read_csv('features_geometry_70.csv')

X = df_data.drop(columns=['class', 'image_name'])
feature_cols = [
    "dist_thumb_index",
    "dist_index_middle",
    "dist_middle_ring",
    "dist_ring_pinky",
    "angle_index",
    "angle_middle",
    "angle_ring",
    "angle_pinky",
]

X = df_data[feature_cols]

y = df_data['class']

#pca = PCA(n_components=N_QUBITS)
#X_pca= pca.fit_transform(X)  

print(X.shape)



le = LabelEncoder()
y_encoded = le.fit_transform(y)

X_cv_raw = df_data[feature_cols].to_numpy()
y_cv = le.fit_transform(df_data['class'].to_numpy())

print(f"Total: {X_cv_raw.shape}")

(45014, 8)
Total: (45014, 8)


In [5]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC


CV_N_SPLITS = 5
SEED = 42
VAL_FRACTION = 0.10

skf = StratifiedKFold(
    n_splits=CV_N_SPLITS,
    shuffle=True,
    random_state=SEED,
)

fold_metrics = []
y_true_all = []
y_pred_all = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_cv_raw, y_cv), start=1):
    print(f"\n===== Fold {fold}/{CV_N_SPLITS} =====")
    start_time = time.time()

    train_idx, _, _, _ = train_test_split(
        train_idx,
        y_cv[train_idx],
        test_size=VAL_FRACTION,
        stratify=y_cv[train_idx],
        random_state=SEED + fold,
    )

    scaler = MinMaxScaler(feature_range=(0, np.pi))

    X_train = scaler.fit_transform(X_cv_raw[train_idx])
    X_test = scaler.transform(X_cv_raw[test_idx])

    y_train = y_cv[train_idx]
    y_test = y_cv[test_idx]

    model = SVC(
        kernel="rbf",
        C=1000,
        decision_function_shape="ovr",
        class_weight="balanced",
        random_state=SEED,
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    elapsed_min = (time.time() - start_time) / 60

    fold_metrics.append({
        "fold": fold,
        "test_acc": acc,
        "test_f1": macro_f1,
        "elapsed_min": elapsed_min,
    })

    y_true_all.extend(y_test.tolist())
    y_pred_all.extend(y_pred.tolist())

    print(
        f"Fold {fold} | "
        f"acc={acc:.4f} | "
        f"f1={macro_f1:.4f} | "
        f"{elapsed_min:.2f} min"
    )




===== Fold 1/5 =====
Fold 1 | acc=0.9979 | f1=0.9979 | 0.02 min

===== Fold 2/5 =====
Fold 2 | acc=0.9976 | f1=0.9976 | 0.02 min

===== Fold 3/5 =====
Fold 3 | acc=0.9981 | f1=0.9981 | 0.01 min

===== Fold 4/5 =====
Fold 4 | acc=0.9980 | f1=0.9980 | 0.01 min

===== Fold 5/5 =====
Fold 5 | acc=0.9974 | f1=0.9974 | 0.01 min


In [6]:
metrics_df = pd.DataFrame(fold_metrics).sort_values("fold")

print(metrics_df.to_string(index=False))
print(
    f"\nAccuracy: {metrics_df['test_acc'].mean():.4f} "
    f"+/- {metrics_df['test_acc'].std(ddof=1):.4f}"
)
print(
    f"Macro F1: {metrics_df['test_f1'].mean():.4f} "
    f"+/- {metrics_df['test_f1'].std(ddof=1):.4f}"
)

print(
    classification_report(
        y_true_all,
        y_pred_all,
        target_names=le.classes_,
        digits=4,
        zero_division=0,
    )
)

 fold  test_acc  test_f1  elapsed_min
    1  0.997890 0.997896     0.015832
    2  0.997556 0.997571     0.015025
    3  0.998112 0.998103     0.013704
    4  0.998001 0.998024     0.013683
    5  0.997445 0.997446     0.013701

Accuracy: 0.9978 +/- 0.0003
Macro F1: 0.9978 +/- 0.0003
              precision    recall  f1-score   support

           A     0.9972    0.9938    0.9955      2113
           B     0.9915    1.0000    0.9957      2224
           C     0.9994    0.9988    0.9991      1662
           D     0.9991    0.9986    0.9989      2200
           E     0.9996    0.9991    0.9993      2243
           F     0.9947    0.9981    0.9964      2069
           G     0.9986    0.9991    0.9989      2200
           I     1.0000    1.0000    1.0000      2200
           L     1.0000    1.0000    1.0000      2200
           M     0.9986    0.9908    0.9947      2169
           N     0.9948    0.9948    0.9948      2120
           O     0.9989    0.9989    0.9989      1868
           P